# Analysis of gene expression, cbdb

In [ ]:
# run in py_decoupler

In [ ]:
import matplotlib.pyplot as plt
import tqdm as notebook_tqdm
from datetime import date
import seaborn as sns
import pandas as pd
import scanpy as sc
import numpy as np
import scipy
import sys
import os
import re
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
#import decoupler as dc
import decoupler as dc, omnipath as op


import palantir
import matplotlib as mpl


#import src.visualization.scBasic as kvis

f"Last execution: {date.today()}"

In [ ]:
from matplotlib import rcParams

In [ ]:
print(dc.__version__) 
print(np.__version__)
print(sc.__version__)
print(sys.version)

## Set input and output directories

In [ ]:
# -- Set base directory
base_dir = '/nfs/team292/rs40/projects/Aneuploid_screen_v2'
sys.path.insert(1, base_dir)
os.chdir(base_dir)

pd.set_option("display.max_columns", 50)
%matplotlib inline

In [ ]:
#meta file 
#meta = '/nfs/team292/rs40/projects/Aneuploid_screen/processed_data/meta_integrated.csv'

In [ ]:
# -- Outputs
#datafiles
output_dir = base_dir+'/processed_data/11_geneexpression_xclone'
path = os.path.join(base_dir, output_dir)

if not os.path.exists(path):
    os.makedirs(path)

#figures
figure_dir = output_dir


## Default scanpy settings

In [ ]:
sc.settings.figdir = path


plt.rcParams.update({
    "figure.figsize": (3, 3),    # Default figure size
    "figure.dpi": 300,           # High resolution
    "font.size": 7,              # Global font size (fallback)
    "axes.titlesize": 7,        # Title size (e.g., 'Leiden')
    "axes.labelsize": 7,         # Axis labels (e.g., 'UMAP1')
    "xtick.labelsize": 7,        # Tick numbers on X axis & Colorbars
    "ytick.labelsize": 7,        # Tick numbers on Y axis & Colorbars
    "legend.fontsize": 7,        # Legend text
    "lines.markersize": 1,       # Dot size in legends
    "axes.spines.top": False,    # Remove top border globally
    "axes.spines.right": False   # Remove right border globally
})


sc.settings.figdir = output_dir

sc.set_figure_params(
    scanpy=True,           # Use Scanpy's opinionated style defaults
    dpi=300,               # High resolution for publication
    dpi_save=300,          # Resolution for saved files
    frameon=True,         # Remove box around plots (cleaner)
    vector_friendly=False,  # usage for PDF/SVG editors (Illustrator)
    fontsize=7,            # Set the base font size (very small for 3-inch figures)
    figsize=(3, 3),        # Set default figure size
    #facecolor=None,     # Ensure background is white (not transparent)
    format='pdf'           # Default save format
)

plt.rcParams['xtick.labelsize'] = 7
plt.rcParams['ytick.labelsize'] = 7

# This is the critical setting for Adobe Illustrator
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

matplotlib.rcParams['svg.fonttype'] = 'none' 
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans"]  # ships with Matplotlib


In [ ]:
celltype_colors = {
    "early EPI": "#E14A5B",
    "Prelineage": "#FBD8EC",  # Assuming '#premorula' corresponds to '#FBD8EC'
    "Morula": "#D74426",
    "late EPI": "#AE3B36",
    "premorula": "#FBD8EC",
    "ECTO": "#FF80A9",
    "EPI.PrE.INT": "#E1529D",
    "AdvMes": "#AB4979",
    "Mesoderm": "#E1529D",
    "ExE_Mes": "#D82679",
    "PriS": "#A777CD",
    "YSE": "#690FA0",
    "Hypoblast": "#7B36FF",
    "early TE": "#B3D51C",
    "TE": "#47A79B",
    "late TE": "#49B91B",
    "polar TE": "#A79E33",
    "CTB": "#418A7D",
    "STB": "#508E36",
    "EVT": "#849D2C",
    "DE": "#00AAD6",
    "doublet": "#BBBBBB",
    "unknown": "#676767"
}

celltype_colors = {
    "naive EPI":    "#F8B949",  # warm yellow
    "blastoid EPI": "#E79033",  # orange
    "TE":       "#3F77C5",  # blue
    "unspecified":       "#676767",  # ggrey
}

In [ ]:
condition_colors2 = {
    "Control":   "#6DAE4E",  # dark grey
    "Mosaic":    "#F2B84B",  # orange
    "Reversine": "#C23A75",  # purple
}

In [ ]:
custom_palette = [
  "#D3D3D4","#555e7b", "#b7d968", "#b576ad", "#e04644", "#fde47f", "#7ccce5", 
  "#C6E5D9", "#F0A830", "#e04644", "#C0D860", "#F2F26F", "#A8E6CE", 
  "#CCC68D", "#EB6841", "#E1F5C4", "#D9CEB2", "#C5CEAE", "#E84A5F", 
  "#A0C55F", "#DCE9BE", "#FFAAA6", "#F07818", "#E08E79", "#A0C55F", 
  "#948C75", "#C0D860", "#005F6B", "#45484B", "#0B2E59", "#FFF7BD", 
  "#CFBE27", "#F1D4AF", "#C02942", "#5E412F", "#355C7D", "#F27435", 
  "#AAB3AB", "#4ECDC4", "#8C2318", "#FF9E9D", "#E6AC27", "#C7F464", 
  "#4ECDC4", "#ED303C", "#F4FAD2", "#F07818", "#031634", "#838689", 
  "#73626E", "#F9D423", "#C06C84", "#F04155", "#F5634A", "#DFBA69", 
  "#A0C55F"
]

In [ ]:
aneu_col = { "monosomy": "#109E9D",
        "trisomy": "#F26B3B",
        'complex':"#662C91", 
        "diploid": "#C5C5C5"}

In [ ]:
white_to_magenta = LinearSegmentedColormap.from_list(
    "white_to_891753", ["#FFFFFF", "#891753"]
)

white_to_cyan = LinearSegmentedColormap.from_list(
    "white_to_cyan", ["#FFFFFF", "#148991"]
)

## Load data

In [ ]:
adata = sc.read_h5ad("/nfs/team292/rs40/projects/Aneuploid_screen_v2/processed_data/3_scanpy_integration/integrated_sub_adata.h5ad")

In [ ]:
adata.obs

In [ ]:
adata.obs.columns

In [ ]:
adata

from scipy.io import mmwrite

# 1. Subset the adata to only highly variable genes
# This checks the 'highly_variable' column in adata.var
#adata_hvg = adata[:, adata.var['highly_variable']].copy()

adata_hvg = adata

print(f"Original shape: {adata.shape}")
print(f"HVG subset shape: {adata_hvg.shape}")

# 2. Save the Metadata (obs) - use the same barcodes
adata_hvg.obs.to_csv(output_dir + "/metadata.csv")

# 3. Save the HVG Gene Names (var)
# These will be the top informative genes
pd.DataFrame(adata_hvg.var_names).to_csv(output_dir + "/genes_hvg.csv", index=False)

# 4. Save the UMAP coordinates
umap_df = pd.DataFrame(adata_hvg.obsm['X_umap'], index=adata_hvg.obs_names, columns=['UMAP1', 'UMAP2'])
umap_df.to_csv(output_dir + "/umap_coords.csv")

# 5. Save the Sparse Matrix (Transposed for R)
mmwrite(output_dir + "/matrix_hvg.mtx", adata_hvg.X.T)

In [ ]:
title = "celltype"
idx = np.random.permutation(adata.n_obs)


with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (1.5, 1.5),
    "figure.dpi": 300,
    "axes.titlesize": 7,
    "legend.fontsize": 7,
    "axes.labelsize": 7,
}):
    sc.pl.umap(
        adata[idx, :],
        color="integrated_celltype",
        size=1,
        palette=celltype_colors,
        #edgecolor="none",
        linewidth=0.5,
        #rasterized=False,
        save = title + "_annotation_"+".pdf"
    )


In [ ]:
#plot for QC

#sc.tl.umap(adata, min_dist=1.5, spread=2) #changed these values based on previous plots
with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (3, 3), 
    "figure.dpi": (100)
}):
    QC_plot = sc.pl.umap(adata,
               color = ['sample', "n_genes_by_counts",
                        'phase', "integrated_celltype","celltype_fine", "celltype_coarse"],
               wspace = 0.6,
               ncols = 3,
               alpha = 0.75,
               size = 10,
               save = "_harmony_QC.pdf"
    #            legend_loc = 'on data',
              )


QC_plot

In [ ]:
adata_sub.obs.head()

In [ ]:
adata.obs['Condition'].unique()

In [ ]:
timpoint = "day1"

sub = adata[adata.obs['timepoint'] == timpoint].copy()   # copy avoids view-of-view issues
idx = np.random.permutation(sub.n_obs)
sub = sub[idx, :]

with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (3, 3),
    "figure.dpi": 100,
    'axes.titlesize': 8,
    "legend.fontsize": 8,
    'axes.labelsize': 8,
}):
    ax = sc.pl.umap(adata, size=10, show=False)
    sc.pl.umap(
        sub,
        color=["Condition"],
        ax=ax,
        size=10,
        palette=condition_colors2,
        save=timpoint + "_annotation_" + ".pdf",
    )

In [ ]:
timpoint = "day4"

sub = adata[adata.obs['timepoint'] == timpoint].copy()   # copy avoids view-of-view issues
idx = np.random.permutation(sub.n_obs)
sub = sub[idx, :]

with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (3, 3),
    "figure.dpi": 100,
    'axes.titlesize': 8,
    "legend.fontsize": 8,
    'axes.labelsize': 8,
}):
    ax = sc.pl.umap(adata, size=10, show=False)
    sc.pl.umap(
        sub,
        color=["Condition"],
        ax=ax,
        size=10,
        palette=condition_colors2,
        save=timpoint + "_annotation_" + ".pdf",
    )

In [ ]:
timepoint = "day6"
remove_samples = ["T3_naive_rev", "T3_mix_bad"]

sub = adata[
    (adata.obs["timepoint"] == timepoint) &
    (~adata.obs["sample"].isin(remove_samples))
].copy()

# Randomise plotting order
idx = np.random.permutation(sub.n_obs)
sub = sub[idx, :]

with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (3, 3),
    "figure.dpi": 100,
    "axes.titlesize": 8,
    "legend.fontsize": 8,
    "axes.labelsize": 8,
}):
    ax = sc.pl.umap(adata, size=10, show=False)
    sc.pl.umap(
        sub,
        color=["Condition"],
        ax=ax,
        size=10,
        palette=condition_colors2,
        save=timpoint + "_annotation" + ".pdf",
    )

In [ ]:
#inferCNV
#inferCNV_file = '/nfs/team292/rs40/projects/scRNAseq_aneu_practice01/processed_data/inferCNV/petropoulos/ploidy_cell_petropoulos.csv'

#scploid
ploidy_df = pd.read_csv( "/nfs/team292/rs40/projects/Aneuploid_screen_v2/processed_data/7_2_xclone/cell_karyotype_scploid.csv", index_col=0)


In [ ]:
ploidy_df

In [ ]:
# adata with ploidy info
adata.obs = adata.obs.set_index('cell.ID_').join(ploidy_df.set_index('cell'), how='left', sort=False)

In [ ]:
adata.obs['cellID'] = adata.obs.index

In [ ]:
adata.obs.head()

In [ ]:
adata.obs['ploidy_xclone'] = adata.obs['ploidy']

In [ ]:
BC_df = pd.read_csv('/nfs/team292/rs40/projects/Aneuploid_screen/processed_data/barcode/BC_df.csv')

In [ ]:
BC_df.head()

In [ ]:
BC_df["cellID"] = (
    BC_df["cell.ID_"].astype(str)
    + "_"
    + BC_df["dataset"].astype(str)
)

In [ ]:
adata.obs = pd.merge(
    adata.obs,
    BC_df,
    on="cellID",
    how="left",
    suffixes=("", "_2")
)

adata.obs

In [ ]:
#visualise the clusters so far
#plot for QC
with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (3, 3), 
    "figure.dpi": (100),
    "savefig.dpi" :(300)
}):
    g=sc.pl.umap(adata, color=["ploidy_xclone"],
           #legend_loc="on data",
           ncols = 4,
           legend_fontsize = 'xx-small',
           size = 5,
           alpha = 0.75,
           palette= aneu_col, 
           wspace=0.5,
          save = "_ploidy.pdf")

g

In [ ]:
adata.obs.head()

In [ ]:
adata_sub = adata[adata.obs["Condition"] == "Control"].copy()

In [ ]:
# 1) Run DE for all clusters vs rest
sc.tl.rank_genes_groups(
    adata_sub,
    groupby= "integrated_celltype",
    method="wilcoxon"
)

In [ ]:
adata_sub.uns["integrated_celltype_colors"]

In [ ]:
# Colours must follow the exact category order
adata_sub.uns["integrated_celltype_colors"] = [
    celltype_colors[x]
    for x in adata_sub.obs["integrated_celltype"].cat.categories
]

In [ ]:
# 2) Heatmap of top N DE genes per cluster

with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.dpi": (300),
    "axes.linewidth": 0.5
}):
    sc.pl.rank_genes_groups_heatmap(
        adata_sub,
        groupby="integrated_celltype",
        n_genes=10,        # number of top genes per cluster
        standard_scale="var",  # z-score per gene
        #swap_axes=True,        # genes on y, clusters on x (often nicer)
        show_gene_labels=True,
        dendrogram=False,
        figsize=(6, 2),
        var_group_positions=None,     
        cmap='viridis',
        save = "sub_DE_heatmap.pdf"
    )

### Categorise chr in good and bad 

In [ ]:
adata.obs['karyotype'].unique()

In [ ]:
#Top and Bottom 20% CLR
developed_pattern = r'(^|,\s*)(13\+|22\+|14\+|19\+|15\+|9\+|6\+|11-)(?=,|$)'
failed_pattern    = r'(^|,\s*)(12\+|12-|19-|6-|17-|22-|21-|20-)(?=,|$)'


dev_hit = adata.obs['karyotype'].str.contains(developed_pattern, regex=True, na=False)
fail_hit = adata.obs['karyotype'].str.contains(failed_pattern, regex=True, na=False)

is_euploid = adata.obs['ploidy_xclone'].eq('diploid')

adata.obs['CLR enrichment'] = np.select(
    [
        is_euploid,
        dev_hit & ~fail_hit,
        fail_hit & ~dev_hit
    ],
    [
        'euploid',
        'Developed_enriched',
        'Failed_enriched'
    ],
    default='others_aneuploid'
)

adata.obs['CLR enrichment'] = pd.Categorical(
    adata.obs['CLR enrichment'],
    categories=[ 'euploid', 'Developed_enriched', 'Failed_enriched', 'others_aneuploid'],
    ordered=True
)

In [ ]:
#Top and Bottom 1/3 trisomy CLR
developed_pattern = r'(^|,\s*)(13\+|22\+|14\+|19\+|15\+|9\+|6\+)(?=,|$)'
failed_pattern    = r'(^|,\s*)(12\+|5\+|7\+|2\+|4\+|20\+|17\+)(?=,|$)'


is_euploid = adata.obs['ploidy_xclone'].eq('diploid')
is_trisomy = adata.obs['ploidy_xclone'].eq('trisomy')


dev_hit = (
    is_trisomy &
    adata.obs['karyotype'].str.contains(developed_pattern, regex=True, na=False)
)

fail_hit = (
    is_trisomy &
    adata.obs['karyotype'].str.contains(failed_pattern, regex=True, na=False)
)


adata.obs['CLR enrichment_Trisomy'] = np.select(
    [
        is_euploid,
        dev_hit & ~fail_hit,
        fail_hit & ~dev_hit
    ],
    [
        'euploid',
        'Developed_enriched',
        'Failed_enriched'
    ],
    default='others_aneuploid'
)

adata.obs['CLR enrichment_Trisomy'] = pd.Categorical(
    adata.obs['CLR enrichment_Trisomy'],
    categories=[ 'euploid', 'Developed_enriched', 'Failed_enriched', 'others_aneuploid'],
    ordered=True
)

In [ ]:
adata.obs[adata.obs['ploidy_xclone'] =='complex']

In [ ]:
adata.obs["ploidy_coarse"] = np.where(
    adata.obs["ploidy_xclone"] == "diploid",
    "euploid",
    "aneuploid"
)

# Fraction of cycling cells

In [ ]:
adata.obs["cycling"] = adata.obs["phase"].isin(["S", "G2M"])

cycling_summary = (
    adata.obs
    .groupby(["celltype_coarse"], observed=True)
    .agg(
        n_cells=("cycling", "size"),
        n_cycling=("cycling", "sum"),
        cycling_fraction=("cycling", "mean"),
        mean_S_score=("S_score", "mean"),
        mean_G2M_score=("G2M_score", "mean")
    )
    .reset_index()
)
cycling_summary

In [ ]:
adata.obs["cycling"] = adata.obs["phase"].isin(["S", "G2M"])

cycling_summary = (
    adata.obs
    .groupby(["celltype_coarse"], observed=True)
    .agg(
        n_cells=("cycling", "size"),
        n_cycling=("cycling", "sum"),
        cycling_fraction=("cycling", "mean"),
        mean_S_score=("S_score", "mean"),
        mean_G2M_score=("G2M_score", "mean")
    )
    .reset_index()
)
cycling_summary

In [ ]:
cycling_summary = (
    adata.obs
    .groupby(["celltype_coarse"], observed=True)
    .agg(
        n_cells=("cycling", "size"),
        n_cycling=("cycling", "sum"),
        cycling_fraction=("cycling", "mean"),
        mean_S_score=("S_score", "mean"),
        mean_G2M_score=("G2M_score", "mean")
    )
    .reset_index()
)
cycling_summary

## Specific Genes

In [ ]:
markers = {'celltype': ['LAMA4', 'LEF1','NANOG', 'GATA3', 'GATA6' ],
           'implantation':["BSG"],
             'immunity':['C3','CD46', 'CD55','CD276','CD47'],
          'adhesion' :['CDH1', 'ITGB1', 'ITGA6','ITGAV', 'ITGB5']}

In [ ]:
markers = {  'adhesion' :['CDH1', 'ITGB1', 'ITGA6','ITGAV', 'ITGB5']}

In [ ]:
dp = sc.pl.dotplot(
    adata, markers, groupby="celltype_coarse",
    cmap=white_to_cyan,
    swap_axes=True,          # puts genes on x-axis
    return_fig=True, show=False
)
ax = dp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)

In [ ]:
# plot for Day0
dp = sc.pl.dotplot(
    adata[adata.obs["timepoint"] == "day1"], markers, groupby="ploidy_xclone",
    cmap=white_to_cyan,
    swap_axes=True,          # puts genes on x-axis
    return_fig=True, show=False
)
ax = dp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)

dp.fig.set_size_inches(5, 5)            # (width, height) in inches
dp.fig.tight_layout()                   # tidy spacing

# Save (vector PDF) — crisp in papers
dp.fig.savefig(path +"/dotplot_D0_ploidy.pdf", bbox_inches="tight")

In [ ]:
sc.pl.violin(
    adata[adata.obs["timepoint"] == "day1"],
    keys="CDH1",
    groupby="ploidy_xclone",     # or "celltype_coarse"
    jitter=0.4,
    stripplot=True,
    rotation=90
)

In [ ]:
# for Day4

ploidy_order = ["euploid", "trisomy", "monosomy", "complex"]

sub = adata[adata.obs["timepoint"] == "day4"].copy()
sub.obs["panel"] = pd.Categorical(
    sub.obs["ploidy_xclone"].astype(str),
    categories=ploidy_order,
    ordered=True
)

# build combined group label: "<ploidy> | <annotation>"
sub.obs["grp"] = sub.obs["celltype_coarse"].str.cat(sub.obs["panel"].astype(str), sep=" | ")

# order rows by ploidy block, then annotation
order = (sub.obs
         .sort_values(["celltype_coarse", "panel"])
         ["grp"].drop_duplicates().tolist())
sub.obs["grp"] = pd.Categorical(sub.obs["grp"], categories=order, ordered=True)


# prevent implicit displays inside this block
# --- build the dotplot (no auto-show) ---
dp = sc.pl.dotplot(
    sub, markers, groupby="grp",
    cmap=white_to_cyan, swap_axes=True,
    return_fig=True, show=False
)

# --- edit the ACTUAL axes inside dp ---
ax = dp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)
ax.xaxis.set_ticks_position("top")
ax.xaxis.set_label_position("top")
ax.set_title("Day4", fontsize=12)

# size & layout
dp.fig.set_size_inches(12, 5)
dp.fig.subplots_adjust(top=0.90)     # extra room for top labels


# save 
out_file = path + "/dotplot_D4_ploidy_by_annotation.pdf"
dp.fig.savefig(out_file, bbox_inches="tight")
    
# plt
display(dp.fig)
plt.close(dp.fig)

In [ ]:
# for Day6

ploidy_order = ["euploid", "trisomy", "monosomy", "complex"]

sub = adata[adata.obs["timepoint"] == "day6"].copy()
sub.obs["panel"] = pd.Categorical(
    sub.obs["ploidy_xclone"].astype(str),
    categories=ploidy_order,
    ordered=True
)

# build combined group label: "<ploidy> | <annotation>"
sub.obs["grp"] = sub.obs["celltype_coarse"].str.cat(sub.obs["panel"].astype(str), sep=" | ")

# order rows by ploidy block, then annotation
order = (sub.obs
         .sort_values(["celltype_coarse", "panel"])
         ["grp"].drop_duplicates().tolist())
sub.obs["grp"] = pd.Categorical(sub.obs["grp"], categories=order, ordered=True)


# prevent implicit displays inside this block
# --- build the dotplot (no auto-show) ---
dp = sc.pl.dotplot(
    sub, markers, groupby="grp",
    cmap=white_to_cyan, swap_axes=True,
    return_fig=True, show=False
)

# --- edit the ACTUAL axes inside dp ---
ax = dp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)
ax.xaxis.set_ticks_position("top")
ax.xaxis.set_label_position("top")
ax.set_title("Day6", fontsize=12)

# size & layout
dp.fig.set_size_inches(12, 5)
dp.fig.subplots_adjust(top=0.90)     # extra room for top labels


# save 
out_file = path + "/dotplot_D6_ploidy_by_annotation.pdf"
dp.fig.savefig(out_file, bbox_inches="tight")
    
# plt
display(dp.fig)
plt.close(dp.fig)

In [ ]:
#apoptotic
markers = {'core': ["CASP3","CASP7","DFFA","DFFB"],
           'TP53':["PMAIP1","BBC3","TP53"],
             'NF-κB':['C3','CD46', 'CD55','CD276','CD47'],
          'death receptor' :["FAS","FASLG","TNFSF10","TNFRSF10A","TNFRSF10B","FADD","CASP8","CASP10","CFLAR","BID"],
          'Phagocytic-opsonin':["MFGE8","GAS6","PROS1","C1QA","C1QC","C3","THBS1","PTX3"],
          'PtdSer' :["XKR8","XKR4","XKR9","ANO6","ATP11C","ATP11A","TMEM30A","CALR","ANXA1"]}

In [ ]:
dp = sc.pl.dotplot(
    adata, markers, groupby="celltype_coarse",
    cmap=white_to_magenta,
    swap_axes=True,          # puts genes on x-axis
    return_fig=True, show=False
)
ax = dp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)

In [ ]:
# plot for Day0
dp = sc.pl.dotplot(
    adata[adata.obs["timepoint"] == "day1"], markers, groupby="ploidy_xclone",
    cmap=white_to_magenta,
    swap_axes=True,          # puts genes on x-axis
    return_fig=True, show=False
)
ax = dp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)

dp.fig.set_size_inches(5, 10)            # (width, height) in inches
dp.fig.tight_layout()                   # tidy spacing

# Save (vector PDF) — crisp in papers
dp.fig.savefig(path +"/dotplot_D0_ploidy_apoptosis.pdf", bbox_inches="tight")

In [ ]:
# for Day4

ploidy_order = ["euploid", "trisomy", "monosomy", "complex"]

sub = adata[adata.obs["timepoint"] == "day4"].copy()
sub.obs["panel"] = pd.Categorical(
    sub.obs["ploidy_xclone"].astype(str),
    categories=ploidy_order,
    ordered=True
)

# build combined group label: "<ploidy> | <annotation>"
sub.obs["grp"] = sub.obs["celltype_coarse"].str.cat(sub.obs["panel"].astype(str), sep=" | ")

# order rows by ploidy block, then annotation
order = (sub.obs
         .sort_values(["celltype_coarse", "panel"])
         ["grp"].drop_duplicates().tolist())
sub.obs["grp"] = pd.Categorical(sub.obs["grp"], categories=order, ordered=True)


# prevent implicit displays inside this block
# --- build the dotplot (no auto-show) ---
dp = sc.pl.dotplot(
    sub, markers, groupby="grp",
    cmap=white_to_magenta, swap_axes=True,
    return_fig=True, show=False
)

# --- edit the ACTUAL axes inside dp ---
ax = dp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)
ax.xaxis.set_ticks_position("top")
ax.xaxis.set_label_position("top")
ax.set_title("Day4", fontsize=12)

# size & layout
dp.fig.set_size_inches(12, 10)
dp.fig.subplots_adjust(top=0.90)     # extra room for top labels


# save 
out_file = path + "/dotplot_D4_apoptosis.pdf"
dp.fig.savefig(out_file, bbox_inches="tight")
    
# plt
display(dp.fig)
plt.close(dp.fig)

In [ ]:
# for Day6

ploidy_order = ["euploid", "trisomy", "monosomy", "complex"]

sub = adata[adata.obs["timepoint"] == "day6"].copy()
sub.obs["panel"] = pd.Categorical(
    sub.obs["ploidy_xclone"].astype(str),
    categories=ploidy_order,
    ordered=True
)

# build combined group label: "<ploidy> | <annotation>"
sub.obs["grp"] = sub.obs["celltype_coarse"].str.cat(sub.obs["panel"].astype(str), sep=" | ")

# order rows by ploidy block, then annotation
order = (sub.obs
         .sort_values(["celltype_coarse", "panel"])
         ["grp"].drop_duplicates().tolist())
sub.obs["grp"] = pd.Categorical(sub.obs["grp"], categories=order, ordered=True)


# prevent implicit displays inside this block
# --- build the dotplot (no auto-show) ---
dp = sc.pl.dotplot(
    sub, markers, groupby="grp",
    cmap=white_to_magenta, swap_axes=True,
    return_fig=True, show=False
)

# --- edit the ACTUAL axes inside dp ---
ax = dp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)
ax.xaxis.set_ticks_position("top")
ax.xaxis.set_label_position("top")
ax.set_title("Day6", fontsize=12)

# size & layout
dp.fig.set_size_inches(12, 10)
dp.fig.subplots_adjust(top=0.90)     # extra room for top labels


# save 
out_file = path + "/dotplot_D6_apoptosis.pdf"
dp.fig.savefig(out_file, bbox_inches="tight")
    
# plt
display(dp.fig)
plt.close(dp.fig)

## Differential expression

In [ ]:
adata.uns['log1p']["base"] = None

In [ ]:
adata.obs['ploidy_xclone']

In [ ]:
adata.obs['ploidy_xclone'] = adata.obs['ploidy_xclone'].cat.reorder_categories(['diploid', 'trisomy','monosomy','complex'])

In [ ]:
adata.obs["celltype_coarse"] = adata.obs["celltype_coarse"].astype(str)

In [ ]:
adata.obs['celltype_coarse'].unique()

In [ ]:
adata.obs["celltype_category"] = adata.obs['celltype_coarse'].apply(
    lambda x: 
    "TE" if "TE" in x or "STB" in x else 
    "EPI" if "EPI" in x 
    else x
)

In [ ]:
adata.obs["celltype_category"].unique()

In [ ]:
# By ploidy type

for cell_type in adata.obs["celltype_category"].unique() :
    adata_sub = adata[adata.obs["celltype_category"] == cell_type]
    sc.tl.rank_genes_groups(adata_sub, groupby = "ploidy_xclone")
    #sc.pl.rank_genes_groups_heatmap(adata_sub, groupby="ploidy_xclone", n_genes=5, save="_ploidy_"+cell_type, dendrogram=False)
    sc.pl.rank_genes_groups_matrixplot(adata_sub, groupby="ploidy_xclone", n_genes=5, save="_ploidy_"+cell_type, 
                                       dendrogram=False, 
                                       title = cell_type, 
                                       #swap_axes = True, 
                                       vmax = 2.5

)

### by CLR category

In [ ]:

title = "day4_CLR_developed_vs_failed_and others"

adata_sub = adata[adata.obs["timepoint"] == "day4"]
sc.tl.rank_genes_groups(adata_sub, groupby = 'CLR enrichment')
#sc.pl.rank_genes_groups_heatmap(adata_sub, groupby="ploidy_xclone", n_genes=5, save="_ploidy_"+cell_type, dendrogram=False)
sc.pl.rank_genes_groups_matrixplot(adata_sub, groupby='CLR enrichment', n_genes=5, save=title , 
                                   dendrogram=False, 
                                   title = "", 
                                   #swap_axes = True, 
                                   #vmax = 2.5,
                                   standard_scale="var"
                            )

In [ ]:
title = "day4_CLR_developed_vs_failed"

adata_sub = adata[
    (adata.obs["timepoint"] == "day4") &
    (adata.obs["CLR enrichment"].isin(["Developed_enriched", "Failed_enriched"]))
].copy()


sc.tl.rank_genes_groups(adata_sub, groupby = 'CLR enrichment')
#sc.pl.rank_genes_groups_heatmap(adata_sub, groupby="ploidy_xclone", n_genes=5, save="_ploidy_"+cell_type, dendrogram=False)
sc.pl.rank_genes_groups_matrixplot(adata_sub, groupby='CLR enrichment', n_genes=10, save=title , 
                                   dendrogram=False, 
                                   title = "",
                                    groups=["Developed_enriched", 'Failed_enriched'],
                                   #swap_axes = True, 
                                   #vmax = 2.5,
                                   #standard_scale="var"
                            )

In [ ]:
title = "day4_CLR_developed_vs_failed_TRI"

adata_sub = adata[
    (adata.obs["timepoint"] == "day4") &
    (adata.obs["CLR enrichment_Trisomy"].isin(["Developed_enriched", "Failed_enriched", 'euploid']))
].copy()


sc.tl.rank_genes_groups(adata_sub, groupby = 'CLR enrichment_Trisomy')
#sc.pl.rank_genes_groups_heatmap(adata_sub, groupby="ploidy_xclone", n_genes=5, save="_ploidy_"+cell_type, dendrogram=False)
sc.pl.rank_genes_groups_matrixplot(adata_sub, groupby='CLR enrichment_Trisomy', n_genes=10, save=title , 
                                   dendrogram=False, 
                                   title = "",
                                    groups=["Developed_enriched", 'Failed_enriched', 'euploid'],
                                   #swap_axes = True, 
                                   #vmax = 2.5,
                                   standard_scale="var"
                            )

In [ ]:
title = "day4_CLR_developed_vs_failed_TRI"

adata_sub = adata[
    (adata.obs["timepoint"] == "day4") &
    (adata.obs["CLR enrichment_Trisomy"].isin(["Developed_enriched", "Failed_enriched"]))
].copy()


sc.tl.rank_genes_groups(adata_sub, groupby = 'CLR enrichment_Trisomy')
#sc.pl.rank_genes_groups_heatmap(adata_sub, groupby="ploidy_xclone", n_genes=5, save="_ploidy_"+cell_type, dendrogram=False)
sc.pl.rank_genes_groups_matrixplot(adata_sub, groupby='CLR enrichment_Trisomy', n_genes=10, save=title , 
                                   dendrogram=False, 
                                   title = "",
                                    groups=["Developed_enriched", 'Failed_enriched'],
                                   #swap_axes = True, 
                                   #vmax = 2.5,
                                   #standard_scale="var"
                            )

## Pathway activity scoring

An alternative approach is to simply score the activity of a pathway or gene signature, in absolute sense, in individual cells, rather than testing for a differential activity between conditions.

### mlm on PROGENy

PROGENy is a comprehensive resource containing a curated collection of pathways and their target genes, with weights for each interaction. For this example we will use the human weights (other organisms are available) and we will use the top 500 responsive genes ranked by p-value. Here is a brief description of each pathway:

In [ ]:
progeny = dc.op.progeny(organism="human", top=500)
progeny

In [ ]:
dc.mt.mlm(data=adata, net=progeny)
score = dc.pp.get_obsm(adata=adata, key="score_mlm")

# scores: cells x pathways
scores_df = score.to_df()
scores_df.index.name = "cell"
scores_df.to_csv(output_dir + "/progeny_mlm_scores.csv")

In [ ]:
scores_df.head()

In [ ]:
sc.pl.matrixplot(score, var_names=score.var_names, groupby=['celltype_coarse'], dendrogram=True, standard_scale='var',
                 colorbar_title='Z-scaled scores', cmap='YlGnBu' )

In [ ]:
#day0
hp = sc.pl.matrixplot(score[score.obs["timepoint"] == "day1"], 
                 var_names=score.var_names, 
                 groupby='ploidy_xclone', 
                 dendrogram=False, 
                      standard_scale='var',
                 colorbar_title='Z-scaled scores',
                 cmap='YlGnBu' ,swap_axes=True, return_fig=True, show=False)

ax = hp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)
ax.set_title("Day0", fontsize=12)

hp.fig.set_size_inches(4, 6)
hp.fig.savefig(
    path + "/pathway_D0_ploidy.pdf",
    bbox_inches="tight"
)


In [ ]:
# Work on a copy
score_plot = score[
    (score.obs["timepoint"] == "day1") &
    (score.obs["celltype_coarse"] == "EPI") &
    (score.obs["Condition"].isin(["Control", "Reversine"]))
].copy()

# Combine condition and ploidy
score_plot.obs["condition_ploidy"] = (
    score_plot.obs["ploidy_xclone"].astype(str)
    + "_"
    + score_plot.obs["Condition"].astype(str)
)

# Optional: inspect cell numbers per group
print(score_plot.obs["condition_ploidy"].value_counts())

In [ ]:
hp = sc.pl.matrixplot(
    score_plot,
    var_names=score_plot.var_names,
    groupby="condition_ploidy",
    dendrogram=False,
    standard_scale="var",
    colorbar_title="Z-scaled scores",
    cmap="YlGnBu",
    swap_axes=True,
    return_fig=True,
    show=False
)

ax = hp.get_axes()["mainplot_ax"]

ax.tick_params(
    axis="x",
    top=True,
    labeltop=True,
    bottom=False,
    labelbottom=False
)

ax.set_title("Day 1 EPI", fontsize=12)

hp.fig.set_size_inches(4, 6)

hp.fig.savefig(
    path + "/pathway_day1_EPI_condition_ploidy.pdf",
    bbox_inches="tight"
)

In [ ]:
import matplotlib as mpl

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"


# subset day0
score_d0 = score[score.obs["timepoint"] == "day1"].copy()

# remove unused categories if categorical
if isinstance(score_d0.obs["ploidy_xclone"].dtype, pd.CategoricalDtype):
    score_d0.obs["ploidy_xclone"] = score_d0.obs["ploidy_xclone"].cat.remove_unused_categories()

# group order
if isinstance(score_d0.obs["ploidy_xclone"].dtype, pd.CategoricalDtype):
    group_order = [g for g in score_d0.obs["ploidy_xclone"].cat.categories
                   if g in score_d0.obs["ploidy_xclone"].unique()]
else:
    group_order = sorted(score_d0.obs["ploidy_xclone"].unique())

# differential test by group
df = dc.tl.rankby_group(
    adata=score_d0,
    groupby="ploidy_xclone",
    reference="diploid",   # change if needed
    method="t-test_overestim_var"
)

# keep only pathways present in score
vars_use = list(score_d0.var_names)
df_sub = df[df["name"].isin(vars_use)].copy()

# convert adjusted p-values to stars
def p_to_star(p):
    if pd.isna(p):
        return ploidy_coarse
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return ploidy_coarse

df_sub["stars"] = df_sub["padj"].apply(p_to_star)

# lookup table: rows = pathway, cols = group
star_df = df_sub.pivot(index="name", columns="group", values="stars").reindex(index=vars_use)
star_df = star_df.reindex(columns=group_order)

# matrix plot
hp = sc.pl.matrixplot(
    score_d0,
    var_names=vars_use,
    groupby='ploidy_xclone',
    categories_order=group_order,
    dendrogram=False,
    standard_scale='var',
    colorbar_title='Z-scaled scores',
    cmap='YlGnBu',
    swap_axes=True,
    return_fig=True,
    show=False
)

hp.make_figure()
hp.fig.set_size_inches(4, 6)

ax = hp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)
ax.set_title("Day0", fontsize=12)

# overlay stars
for yi, var in enumerate(vars_use):
    for xi, grp in enumerate(group_order):
        star = ploidy_coarse
        if var in star_df.index and grp in star_df.columns:
            val = star_df.loc[var, grp]
            if pd.notna(val):
                star = val
        if star != ploidy_coarse:
            ax.text(
                xi + 0.5, yi + 0.5, star,
                ha="center", va="center",
                color="black", fontsize=8
            )

hp.fig.savefig(
    path + "/pathway_D0_ploidy.pdf",
    bbox_inches="tight"
)

In [ ]:
# for Day4
ploidy_order = ["euploid", "trisomy", "monosomy", "complex"]

sub = score[score.obs["timepoint"] == "day4"].copy()

sub.obs["panel"] = pd.Categorical(
    sub.obs["ploidy_xclone"].astype(str),
    categories=ploidy_order,
    ordered=True
)

# build combined group label: "<ploidy> | <annotation>"
sub.obs["grp"] = sub.obs["celltype_coarse"].str.cat(sub.obs["panel"].astype(str), sep=" | ")

# order rows by ploidy block, then annotation
order = (sub.obs
         .sort_values(["celltype_coarse", "panel"])
         ["grp"].drop_duplicates().tolist())
sub.obs["grp"] = pd.Categorical(sub.obs["grp"], categories=order, ordered=True)


# prevent implicit displays inside this block
# --- build the dotplot (no auto-show) ---
hp = sc.pl.matrixplot(sub, 
                 var_names=score.var_names, 
                 groupby='grp', 
                 dendrogram=False, standard_scale='var',
                 colorbar_title='Z-scaled scores',
                 cmap='YlGnBu' ,swap_axes=True, return_fig=True, show=False)

# --- edit the ACTUAL axes inside dp ---
ax = hp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)
ax.xaxis.set_ticks_position("top")
ax.xaxis.set_label_position("top")
ax.set_title("Day4", fontsize=12)

# size & layout
hp.fig.set_size_inches(9, 5)
hp.fig.subplots_adjust(top=0.90)     # extra room for top labels


# save 
out_file = path + "/pathway_D4_progeny.pdf"
hp.fig.savefig(out_file, bbox_inches="tight")
    
# plt
display(hp.fig)
plt.close(hp.fig)

In [ ]:
# for Day6
ploidy_order = ["euploid", "trisomy", "monosomy", "complex"]

sub = score[score.obs["timepoint"] == "day6"].copy()

sub.obs["panel"] = pd.Categorical(
    sub.obs["ploidy_xclone"].astype(str),
    categories=ploidy_order,
    ordered=True
)

# build combined group label: "<ploidy> | <annotation>"
sub.obs["grp"] = sub.obs["celltype_coarse"].str.cat(sub.obs["panel"].astype(str), sep=" | ")

# order rows by ploidy block, then annotation
order = (sub.obs
         .sort_values(["celltype_coarse", "panel"])
         ["grp"].drop_duplicates().tolist())
sub.obs["grp"] = pd.Categorical(sub.obs["grp"], categories=order, ordered=True)


# prevent implicit displays inside this block
# --- build the dotplot (no auto-show) ---
hp = sc.pl.matrixplot(sub, 
                 var_names=score.var_names, 
                 groupby='grp', 
                 dendrogram=False, standard_scale='var',
                 colorbar_title='Z-scaled scores',
                 cmap='YlGnBu' ,swap_axes=True, return_fig=True, show=False)

# --- edit the ACTUAL axes inside dp ---
ax = hp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)
ax.xaxis.set_ticks_position("top")
ax.xaxis.set_label_position("top")
ax.set_title("Day6", fontsize=12)

# size & layout
hp.fig.set_size_inches(9, 5)
hp.fig.subplots_adjust(top=0.90)     # extra room for top labels


# save 
out_file = path + "/pathway_D6_progeny.pdf"
hp.fig.savefig(out_file, bbox_inches="tight")
    
# plt
display(hp.fig)
plt.close(hp.fig)

In [ ]:
#day4 CLR
hp = sc.pl.matrixplot(score[(score.obs["timepoint"] == "day4") & (score.obs['CLR enrichment'] != "others_aneuploid")], 
                 var_names=score.var_names, 
                 groupby='CLR enrichment', 
                 dendrogram=False, standard_scale='var',
                 colorbar_title='Z-scaled scores',
                 cmap='YlGnBu' ,swap_axes=True, return_fig=True, show=False)

ax = hp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)
ax.set_title("Day4_PROGENy_score")

hp.fig.set_size_inches(3, 5)            # (width, height) in inches
hp.fig.tight_layout()                   # tidy spacing


# Save (vector PDF) — crisp in papers
hp.fig.savefig(path +"/pathway_D0_ploidy_all.pdf", bbox_inches="tight")




In [ ]:
#day0
hp = sc.pl.matrixplot(score[score.obs["timepoint"] == "day4"], 
                 var_names=score.var_names, 
                 groupby='CLR enrichment_Trisomy', 
                 dendrogram=False, standard_scale='var',
                 colorbar_title='Z-scaled scores',
                 cmap='YlGnBu' ,swap_axes=True, return_fig=True, show=False)

ax = hp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)
ax.set_title("Day4_CLR_Tri", fontsize=12)

hp.fig.set_size_inches(3, 5)            # (width, height) in inches
hp.fig.tight_layout()                   # tidy spacing


# Save (vector PDF) — crisp in papers
hp.fig.savefig(path +"/pathway_D0_ploidy_try.pdf", bbox_inches="tight")

### mlm on Hallmark

https://decoupler.readthedocs.io/en/latest/notebooks/scell/rna_sc.html#hallmark-gene-sets

Hallmark gene sets are curated collections of genes that represent specific, well-defined biological states or processes. They are part of MSigDB and were developed to reduce redundancy and improve interpretability compared to older, more overlapping gene set collections

In [ ]:
hallmark = dc.op.hallmark(organism="human")
hallmark

In [ ]:
dc.mt.mlm(data=adata, net=hallmark)
score_hallmark = dc.pp.get_obsm(adata=adata, key="score_mlm")
score_hallmark

In [ ]:
# scores: cells x pathways
score_hallmark_df = score_hallmark.to_df()
score_hallmark_df.index.name = "cell"
score_hallmark_df.to_csv(output_dir + "/hallmark_mlm_scores.csv")

In [ ]:
score_hallmark.var_names

In [ ]:
hallmarks_sub = ['APICAL_JUNCTION', 'APICAL_SURFACE', 'APOPTOSIS',
       'COMPLEMENT', 'DNA_REPAIR', 'E2F_TARGETS',
       'EPITHELIAL_MESENCHYMAL_TRANSITION', 'ESTROGEN_RESPONSE_EARLY',
       'ESTROGEN_RESPONSE_LATE', 'FATTY_ACID_METABOLISM', 'G2M_CHECKPOINT',
       'GLYCOLYSIS', 'HEDGEHOG_SIGNALING', 'HYPOXIA',
       'IL2_STAT5_SIGNALING', 'IL6_JAK_STAT3_SIGNALING',
       'INFLAMMATORY_RESPONSE', 'INTERFERON_ALPHA_RESPONSE',
       'INTERFERON_GAMMA_RESPONSE', 'KRAS_SIGNALING_DN', 'KRAS_SIGNALING_UP',
       'MITOTIC_SPINDLE', 'MTORC1_SIGNALING', 'MYC_TARGETS_V1',
       'MYC_TARGETS_V2', 'NOTCH_SIGNALING',
       'OXIDATIVE_PHOSPHORYLATION', 'P53_PATHWAY', 
       'PEROXISOME', 'PI3K_AKT_MTOR_SIGNALING', 'PROTEIN_SECRETION',
       'REACTIVE_OXYGEN_SPECIES_PATHWAY', 
       'TGF_BETA_SIGNALING', 'TNFA_SIGNALING_VIA_NFKB',
       'UNFOLDED_PROTEIN_RESPONSE', 'UV_RESPONSE_DN',
       'WNT_BETA_CATENIN_SIGNALING']

In [ ]:
hallmarks_sub_2 = ['APICAL_JUNCTION', 
                   'DNA_REPAIR', 
                   'WNT_BETA_CATENIN_SIGNALING',
                   'APOPTOSIS', 'P53_PATHWAY',        'EPITHELIAL_MESENCHYMAL_TRANSITION']


In [ ]:
hallmarks_sub_3 = [
    # Directly elimination-related
    'APOPTOSIS',
    'P53_PATHWAY',

    # Supportive upstream/context pathways
    #intrinsic apoptosis
    'REACTIVE_OXYGEN_SPECIES_PATHWAY',
    'UNFOLDED_PROTEIN_RESPONSE',
    'DNA_REPAIR',
    'HYPOXIA',
    
    #extrinsic 
    'TNFA_SIGNALING_VIA_NFKB',
    'INFLAMMATORY_RESPONSE',
    'INTERFERON_ALPHA_RESPONSE',
    'INTERFERON_GAMMA_RESPONSE',
    'IL6_JAK_STAT3_SIGNALING',
    
    'MYC_TARGETS_V1', 
    'MYC_TARGETS_V2', 'MTORC1_SIGNALING', 'PI3K_AKT_MTOR_SIGNALING'
]

In [ ]:
sc.pl.matrixplot(
    adata=score_hallmark,
    var_names=hallmarks_sub,
    groupby='celltype_coarse',
    dendrogram=True,
    standard_scale="var",
    colorbar_title="Z-scaled score_hallmark",
    cmap="YlOrRd"
)


In [ ]:
#day0
hp = sc.pl.matrixplot(score_hallmark[score_hallmark.obs["timepoint"] == "day1"], 
                 var_names=hallmarks_sub_3, 
                 groupby='ploidy_xclone', 
                 dendrogram=False, 
                 standard_scale='var',
                 colorbar_title='Z-scaled score_hallmark',
                 cmap='RdPu' ,swap_axes=True, return_fig=True, show=False)

ax = hp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)
ax.set_title("Day0", fontsize=12)


hp.fig.set_size_inches(4, 8)
hp.fig.savefig(
    path +"/pathway_D0_hallmark.pdf",
    bbox_inches="tight"
)

In [ ]:
# subset day0
score_d0 = score_hallmark[score_hallmark.obs["timepoint"] == "day1"].copy()

# rank hallmarks by group
df = dc.tl.rankby_group(
    adata=score_d0,
    groupby="ploidy_xclone",
    reference="diploid",   # change if needed
    method="wilcoxon"
)

# keep only hallmarks you want
df_sub = df[df["name"].isin(hallmarks_sub_3)].copy()

# convert p-values to stars
def p_to_star(p):
    if pd.isna(p):
        return ""
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return ""

df_sub["stars"] = df_sub["padj"].apply(p_to_star)

# make lookup table: rows = hallmark, cols = ploidy group
star_df = (
    df_sub.pivot(index="name", columns="group", values="stars")
    .reindex(index=hallmarks_sub_3)
)

# group order from obs
if isinstance(score_d0.obs["ploidy_xclone"].dtype, pd.CategoricalDtype):
    group_order = [g for g in score_d0.obs["ploidy_xclone"].cat.categories
                   if g in score_d0.obs["ploidy_xclone"].unique()]
else:
    group_order = sorted(score_d0.obs["ploidy_xclone"].unique())

star_df = star_df.reindex(columns=group_order)

# plot
hp = sc.pl.matrixplot(
    score_d0,
    var_names=[h for h in hallmarks_sub_3 if h in score_d0.var_names],
    groupby="ploidy_xclone",
    categories_order=group_order,
    dendrogram=False,
    standard_scale="var",
    colorbar_title="Z-scaled score_hallmark",
    cmap="RdPu",
    swap_axes=True,
    return_fig=True,
    show=False
)

hp.make_figure() 

hp.fig.set_size_inches(4, 8)
ax = hp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)
ax.set_title("Day0", fontsize=12)

# overlay stars
vars_use = [h for h in hallmarks_sub_3 if h in score_d0.var_names]
for yi, var in enumerate(vars_use):
    for xi, grp in enumerate(group_order):
        star = ""
        if var in star_df.index and grp in star_df.columns:
            val = star_df.loc[var, grp]
            if pd.notna(val):
                star = val
        if star != "":
            ax.text(
                xi + 0.5, yi + 0.5, star,
                ha="center", va="center",
                color="black", fontsize=8
            )

hp.fig.savefig(
    path + "/pathway_D0_hallmark_significance.pdf",
    bbox_inches="tight"
)

In [ ]:
# for Day4
ploidy_order = ["euploid", "trisomy", "monosomy", "complex"]

sub = score_hallmark[score_hallmark.obs["timepoint"] == "day4"].copy()

sub.obs["panel"] = pd.Categorical(
    sub.obs["ploidy_xclone"].astype(str),
    categories=ploidy_order,
    ordered=True
)

# build combined group label: "<ploidy> | <annotation>"
sub.obs["grp"] = sub.obs["celltype_coarse"].str.cat(sub.obs["panel"].astype(str), sep=" | ")

# order rows by ploidy block, then annotation
order = (sub.obs
         .sort_values(["celltype_coarse", "panel"])
         ["grp"].drop_duplicates().tolist())
sub.obs["grp"] = pd.Categorical(sub.obs["grp"], categories=order, ordered=True)


# prevent implicit displays inside this block
# --- build the dotplot (no auto-show) ---
hp = sc.pl.matrixplot(sub, 
                 var_names=hallmarks_sub, 
                 groupby='grp', 
                 dendrogram=False, standard_scale='var',
                 colorbar_title='Z-scaled score_hallmark',
                 cmap='RdPu' ,swap_axes=True, return_fig=True, show=False)

# --- edit the ACTUAL axes inside dp ---
ax = hp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)
ax.xaxis.set_ticks_position("top")
ax.xaxis.set_label_position("top")
ax.set_title("Day4", fontsize=12)

# size & layout
hp.fig.set_size_inches(9, 8)
hp.fig.subplots_adjust(top=0.90)     # extra room for top labels


# save 
out_file = path + "/pathway_D4_hallmark.pdf"
hp.fig.savefig(out_file, bbox_inches="tight")
    
# plt
display(hp.fig)
plt.close(hp.fig)

In [ ]:
#day0
hp = sc.pl.matrixplot(score_hallmark[score_hallmark.obs["timepoint"] == "day4"], 
                 var_names=hallmarks_sub, 
                 groupby='CLR enrichment', 
                 dendrogram=False, 
                 standard_scale='var',
                 colorbar_title='Z-scaled score_hallmark',
                 cmap='RdPu' ,swap_axes=True, return_fig=True, show=False)

ax = hp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)
ax.set_title("Day4", fontsize=12)

hp.fig.set_size_inches(8, 8)            # (width, height) in inches
hp.fig.tight_layout()                   # tidy spacing


# Save (vector PDF) — crisp in papers
hp.fig.savefig(path +"/pathway_hallmark_D4_CLR.pdf")

In [ ]:
# subset first
score_sub = score_hallmark[
    (score_hallmark.obs["timepoint"] == "day4") &
    (score_hallmark.obs["CLR enrichment"] != "others_aneuploid")
].copy()

# set order
score_sub.obs["CLR enrichment"] = pd.Categorical(
    score_sub.obs["CLR enrichment"],
    categories=['euploid', 'Developed_enriched', 'Failed_enriched'],
    ordered=True
)


hp = sc.pl.matrixplot(score_sub, 
                 var_names=hallmarks_sub_2, 
                 groupby='CLR enrichment', 
                 dendrogram=False, 
                 standard_scale='var',
                 colorbar_title='Z-scaled score_hallmark',
                 cmap='RdPu' ,swap_axes=True, return_fig=True, show=False)

ax = hp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)
ax.set_title("Day4_hallmark")

hp.fig.set_size_inches(3.5, 4)            # (width, height) in inches
hp.fig.tight_layout()                   # tidy spacing


# Save (vector PDF) — crisp in papers
hp.fig.savefig(path +"/pathway_hallmark_D4_CLR.pdf", bbox_inches="tight")


In [ ]:
hallmarks_sub_3 = [
    # Directly elimination-related
    'APOPTOSIS',
    'P53_PATHWAY'

]

In [ ]:
#day0

score_sub = score_hallmark[score_hallmark.obs["timepoint"].isin(["day4", "day1"])].copy()

hp = sc.pl.matrixplot(
    score_sub,
    var_names=[h for h in hallmarks_sub_3 if h in score_sub.var_names],
    groupby="ploidy_xclone",
    dendrogram=False,
    standard_scale="var",
    colorbar_title="Z-scaled hallmark score",
    cmap="RdPu",
    swap_axes=True,
    return_fig=True,
    show=False
)

ax = hp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)
ax.set_title("Day1 + Day4 hallmark")

hp.fig.set_size_inches(3, 2)
hp.fig.tight_layout()
hp.fig.savefig(path + "/pathway_hallmark_allTP_ploidy.pdf", bbox_inches="tight")

In [ ]:
score_sub.obs["ploidy_coarse"].unique()

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

hallmarks_sub_3 = [
    "APOPTOSIS",
    "P53_PATHWAY"
]

# subset to day1
score_sub = score_hallmark[
    score_hallmark.obs["timepoint"] == "day1"
].copy()

# make sure ploidy grouping exists and is ordered
score_sub.obs["ploidy_coarse"] = pd.Categorical(
    score_sub.obs["ploidy_coarse"],
    categories=["euploid", "aneuploid"],
    ordered=True
)

# extract scores
df_scores = score_sub.to_df()[hallmarks_sub_3].copy()
df_scores["ploidy_coarse"] = score_sub.obs["ploidy_coarse"].values
df_scores["cell_id"] = score_sub.obs_names

# long format
df_long = df_scores.melt(
    id_vars=["cell_id", "ploidy_coarse"],
    value_vars=hallmarks_sub_3,
    var_name="hallmark",
    value_name="score"
).dropna()

# stats
stats = []
for hm in hallmarks_sub_3:
    sub = df_long[df_long["hallmark"] == hm]
    x = sub.loc[sub["ploidy_coarse"] == "euploid", "score"]
    y = sub.loc[sub["ploidy_coarse"] == "aneuploid", "score"]

    if len(x) > 0 and len(y) > 0:
        stat, pval = mannwhitneyu(x, y, alternative="two-sided")
    else:
        pval = np.nan

    stats.append({
        "hallmark": hm,
        "n_euploid": len(x),
        "n_aneuploid": len(y),
        "pvalue": pval
    })

stats_df = pd.DataFrame(stats)

def p_to_stars(p):
    if pd.isna(p):
        return "ns"
    elif p < 1e-4:
        return "****"
    elif p < 1e-3:
        return "***"
    elif p < 1e-2:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

stats_df["sig"] = stats_df["pvalue"].apply(p_to_stars)



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

hallmarks_sub_3 = [
    "APOPTOSIS",
    "P53_PATHWAY"
]

# subset to day1
score_sub = score_hallmark[
    score_hallmark.obs["timepoint"] == "day1"
].copy()

# make sure grouping column exists
score_sub.obs["ploidy_coarse"] = pd.Categorical(
    score_sub.obs["ploidy_coarse"],
    categories=["euploid", "aneuploid"],
    ordered=True
)

# extract scores
df_scores = score_sub.to_df()[hallmarks_sub_3].copy()
df_scores["ploidy_coarse"] = score_sub.obs["ploidy_coarse"].values
df_scores["cell_id"] = score_sub.obs_names

# long format
df_long = df_scores.melt(
    id_vars=["cell_id", "ploidy_coarse"],
    value_vars=hallmarks_sub_3,
    var_name="hallmark",
    value_name="score"
).dropna()

# compute stats and pseudo-log2FC
results = []
global_min = df_long["score"].min()

shift = abs(global_min) + 1e-6 if global_min <= 0 else 0

for hm in hallmarks_sub_3:
    sub = df_long[df_long["hallmark"] == hm]

    eu = sub.loc[sub["ploidy_coarse"] == "euploid", "score"]
    an = sub.loc[sub["ploidy_coarse"] == "aneuploid", "score"]

    mean_eu = eu.mean()
    mean_an = an.mean()

    if len(eu) > 0 and len(an) > 0:
        stat, pval = mannwhitneyu(an, eu, alternative="two-sided")
    else:
        pval = np.nan

    fc = (mean_an + shift) / (mean_eu + shift)
    log2fc = np.log2(fc)

    results.append({
        "hallmark": hm,
        "mean_euploid": mean_eu,
        "mean_aneuploid": mean_an,
        "fold_change": fc,
        "log2FC": log2fc,
        "pvalue": pval,
        "n_euploid": len(eu),
        "n_aneuploid": len(an)
    })

res_df = pd.DataFrame(results)

def p_to_stars(p):
    if pd.isna(p):
        return "ns"
    elif p < 1e-4:
        return "****"
    elif p < 1e-3:
        return "***"
    elif p < 1e-2:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

res_df["sig"] = res_df["pvalue"].apply(p_to_stars)

title = "hallmark_day1_aneuploid_vs_euploid_log2FC"

fig, ax = plt.subplots(figsize=(2.8, 2.8))

x = np.arange(len(res_df))
bars = ax.bar(x, res_df["log2FC"])

ax.axhline(0, linestyle="--", linewidth=0.8, color="black")
ax.set_xticks(x)
ax.set_xticklabels(res_df["hallmark"], rotation=45, ha="right")
ax.set_ylabel("log2FC (aneuploid / euploid)")
ax.set_title("Day1")

ymax = res_df["log2FC"].max()
ymin = res_df["log2FC"].min()
yrange = ymax - ymin if ymax > ymin else 1

for i, row in res_df.iterrows():
    y = row["log2FC"]
    offset = 0.08 * yrange if yrange > 0 else 0.1

    if y >= 0:
        text_y = y + offset
        va = "bottom"
    else:
        text_y = y - offset
        va = "top"

    ax.text(
        i,
        text_y,
        f'{row["sig"]}\np={row["pvalue"]:.2e}',
        ha="center",
        va=va,
        fontsize=8
    )

plt.tight_layout()
plt.savefig(path + f"/{title}.pdf", bbox_inches="tight")
plt.show()

res_df

In [ ]:
# plot
title = "hallmark_day1_euploid_vs_aneuploid"
w = 3.8
h = 2.4
plt.rcParams["figure.figsize"] = (w, h)

g = sns.catplot(
    data=df_long,
    x="ploidy_coarse",
    y="score",
    #hue="ploidy_simple",
    col="hallmark",
    kind="bar",
    errorbar="se",
    sharey=False,
    height=2.4,
    aspect=0.85,
    legend=False
)

for i, hm in enumerate(hallmarks_sub_3):
    ax = g.axes[0, i]
    sub = df_long[df_long["hallmark"] == hm]

    sns.stripplot(
        data=sub,
        x="ploidy_coarse",
        y="score",
        order=["euploid", "aneuploid"],
        color="black",
        size=2,
        alpha=0.35,
        jitter=0.2,
        ax=ax
    )

    row = stats_df[stats_df["hallmark"] == hm].iloc[0]
    ymax = sub["score"].max()
    ymin = sub["score"].min()
    yrange = ymax - ymin if ymax > ymin else 1
    y = ymax + 0.12 * yrange

    ax.plot([0, 0, 1, 1], [y - 0.02*yrange, y, y, y - 0.02*yrange], lw=1, c="black")
    ax.text(
        0.5, y + 0.03 * yrange,
        f'{row["sig"]}\np={row["pvalue"]:.2e}',
        ha="center", va="bottom", fontsize=8
    )

    ax.set_title(hm)
    ax.set_xlabel("ploidy_coarse")
    ax.set_ylabel("MLM score")

plt.tight_layout()
plt.savefig(path + f"/{title}.pdf", bbox_inches="tight")
plt.show()

stats_df

In [ ]:
#day0
hp = sc.pl.matrixplot(score_hallmark[score_hallmark.obs["timepoint"] == "day4"], 
                 var_names=hallmarks_sub, 
                 groupby='CLR enrichment_Trisomy', 
                 dendrogram=False, 
                 standard_scale='var',
                 colorbar_title='Z-scaled score_hallmark',
                 cmap='RdPu' ,swap_axes=True, return_fig=True, show=False)

ax = hp.get_axes()["mainplot_ax"]
ax.tick_params(axis="x", top=True, labeltop=True, bottom=False, labelbottom=False)
ax.set_title("Day4_Tr", fontsize=12)

hp.fig.set_size_inches(8, 8)            # (width, height) in inches
hp.fig.tight_layout()                   # tidy spacing


# Save (vector PDF) — crisp in papers
hp.fig.savefig(path +"/pathway_hallmark_D4_CLR_Tri.pdf")

## TF activity

In [ ]:
net = dc.get_collectri(organism='human', split_complexes=False)
net

In [ ]:
import decoupler as dc
print(dc.__version__)
print(hasattr(dc, "get_collectri"))

In [ ]:
[x for x in dir(dc) if "collect" in x.lower() or "doro" in x.lower() or "resource" in x.lower()]

In [ ]:
dc.run_ulm(
    mat=adata,
    net=net,
    source='source',
    target='target',
    weight='weight',
    use_raw=False
)

In [ ]:
acts = dc.get_acts(adata, obsm_key='ulm_estimate')

In [ ]:
df = dc.rank_sources_groups(acts, groupby='celltype_coarse', reference='rest', method='t-test_overestim_var')

n_markers = 5
source_markers = df.groupby('group').head(n_markers).groupby('group')['names'].apply(lambda x: list(x)).to_dict()
source_markers

sc.pl.matrixplot(acts, source_markers, 'celltype_coarse', dendrogram=True, standard_scale='var',
                 colorbar_title='Z-scaled scores', cmap='RdBu_r')

In [ ]:
adata.obs['sample'].unique()

In [ ]:
adata.obs

## For cellphoneDB

1. differentially expressed genes for euploid, complex, monosomy, trisomy within mosaic day4. 
2. differentially expressed genes for euploid-in euploid embryos, euploid in mosaic embryos, and complex in mosaic embryos 

### 1. within mosaic day4

In [ ]:
#make a df for DEG for different aneuploid cells in EPI cells
#groups=['complex', 'euploid']

adata_sub = adata[
    (adata.obs['timepoint'] == "day4") &
    (adata.obs['sample'].isin(["T2_mix"]))]
adata_sub.obs['ploidy_xclone'].unique()

In [ ]:
min_count =  adata_sub.obs['ploidy_xclone'].value_counts().min()

# Randomly sample min_count cells from each group
balanced_indices = (
    adata_sub.obs
    .groupby('ploidy_xclone', group_keys=False, observed=False)
    .apply(lambda x: x.sample(min_count, random_state=0))
    .index
)

# Subset the AnnData object
adata_sub = adata_sub[balanced_indices].copy()

In [ ]:
adata_sub.obs['ploidy_xclone'].value_counts()

In [ ]:
sc.tl.rank_genes_groups(adata_sub,
                        #reference = "euploid", 
                        #groups=groups, 
                        groupby = "ploidy_xclone")


deg_df_dict = {}

for cell_type in adata_sub.obs['ploidy_xclone'].unique() :
    #sc.pl.rank_genes_groups_heatmap(adata_sub, groupby="ploidy_xclone", n_genes=5, save="_ploidy_"+cell_type, dendrogram=False)
    deg_df_dict[cell_type] = sc.get.rank_genes_groups_df(adata_sub, group = cell_type)
    deg_df_dict[cell_type]  = deg_df_dict[cell_type][(deg_df_dict[cell_type] ['pvals_adj'] < 0.05) & (deg_df_dict[cell_type]['logfoldchanges'] > 0.25)]
    deg_df_dict[cell_type]['cell_type'] = cell_type
    

DEG_df = pd.concat(list(deg_df_dict.values()))

DEG_df

In [ ]:
#reformat and save for cell phone DB
DEG_df_cpdb = pd.concat([DEG_df['cell_type'], DEG_df['names']], axis = 1)

DEG_df_cpdb= DEG_df_cpdb.rename(columns={"names": "genes"})


In [ ]:
DEG_df_cpdb['cell_type']= DEG_df_cpdb['cell_type'].astype('string')

In [ ]:
DEG_df_cpdb.to_csv(output_dir + '/deg.tsv', sep='\t', index=False)

In [ ]:
output_dir + '/deg.tsv'

In [ ]:
adata_sub.obs.to_csv(output_dir +'/cpdb_meta.csv')

In [ ]:
adata_sub.obs.index = adata_sub.obs['cellID']

In [ ]:
adata_sub.write_h5ad(output_dir+"/adata_cpdb" +".h5ad")

In [ ]:
adata_sub.obs

In [ ]:
sc.pl.dotplot(adata,["EFNA1", "EFNA3",  "EFNA4",  "EFNA5",
                     "EPHA1", "EPHA3", "EPHA4","EPHA5", "EPHA7"], groupby=['celltype_coarse', 'ploidy_xclone'], save = "_ephrins.pdf")

## Further group based on ploidy percent

In [ ]:
adata.obs

In [ ]:
#make a df for DEG for different aneuploid cells in EPI cells
#groups=['complex', 'euploid']

adata_sub = adata[
    (adata.obs['celltype_category'] == "unknown") &
    (adata.obs['dataset'].isin(["T2_control", "T2_mix", "T2_rev"]))&
     (~adata.obs['complexity'].isna())]
adata_sub.obs['ploidy_xclone'].unique()

In [ ]:
adata_sub.obs['complexity'].unique()

In [ ]:
adata_sub.obs['ploidy_proportion'] = adata_sub.obs['ploidy_xclone'].astype(str) +"_"+  adata_sub.obs['complexity'].astype(str)

In [ ]:
min_count =  adata_sub.obs['ploidy_proportion'].value_counts().min()

# Randomly sample min_count cells from each group
balanced_indices = (
    adata_sub.obs
    .groupby('ploidy_proportion', group_keys=False, observed=False)
    .apply(lambda x: x.sample(min_count, random_state=0))
    .index
)

# Subset the AnnData object
adata_sub = adata_sub[balanced_indices].copy()

In [ ]:
adata_sub.obs['ploidy_proportion'].value_counts()

In [ ]:
sc.tl.rank_genes_groups(adata_sub,
                        #reference = "euploid", 
                        #groups=groups, 
                        groupby = "ploidy_proportion")


deg_df_dict = {}

for cell_type in adata_sub.obs['ploidy_proportion'].unique() :
    #sc.pl.rank_genes_groups_heatmap(adata_sub, groupby="ploidy_xclone", n_genes=5, save="_ploidy_"+cell_type, dendrogram=False)
    deg_df_dict[cell_type] = sc.get.rank_genes_groups_df(adata_sub, group = cell_type)
    deg_df_dict[cell_type]  = deg_df_dict[cell_type][(deg_df_dict[cell_type] ['pvals_adj'] < 0.05) & (deg_df_dict[cell_type]['logfoldchanges'] > 0.25)]
    deg_df_dict[cell_type]['cell_type'] = cell_type
    

DEG_df = pd.concat(list(deg_df_dict.values()))

DEG_df

In [ ]:
#reformat and save for cell phone DB
DEG_df_cpdb = pd.concat([DEG_df['cell_type'], DEG_df['names']], axis = 1)

DEG_df_cpdb= DEG_df_cpdb.rename(columns={"names": "genes"})


In [ ]:
DEG_df_cpdb['cell_type']= DEG_df_cpdb['cell_type'].astype('string')

In [ ]:
DEG_df_cpdb['cell_type'].dtype

In [ ]:
DEG_df_cpdb.to_csv(output_dir + '/deg.tsv', sep='\t', index=False)

In [ ]:
output_dir + '/deg.tsv'

In [ ]:
adata_sub.obs.to_csv(output_dir +'/cpdb_meta.csv')

In [ ]:
adata_sub.obs.index = adata_sub.obs['cellID']

In [ ]:
del adata_sub._obsm['ora_estimate']
del adata_sub._obsm['ora_pvals']
adata_sub.write_h5ad(output_dir+"/adata_cpdb" +".h5ad")

## Gene set tests

Gene set tests test whether a pathway is enriched, in other words over-represented, in one condition compared to others, say, in healthy donors compared to severe COVID-19 patients in the monocyte population

### Over Representation Analysis

using ORA: a statistical method used to identify pathways or gene sets that are significantly enriched in a subset of genes — for example, those highly expressed or differentially expressed in your experiment.
Compares the DEGs from your analysis with predefined gene sets (pathways) to determine if any of these gene sets are disproportionately represented compared to what would be expected by chance. Statistical tests are used to assess whether the overlap between your DEGs and pathway genes is significant. (which pathway is over represented in DEG?)

In [ ]:
adata

In [ ]:
msigdb_original = dc.op.resource("MSigDB")

In [ ]:
msigdb_original['collection'].unique()

In [ ]:
# Filter by hallmark
msigdb = msigdb_original[msigdb_original['collection']=='hallmark'].copy()

# Remove duplicated entries
msigdb = msigdb[~msigdb.duplicated(['geneset', 'genesymbol'])]
msigdb

In [ ]:
# msigdb likely has columns: geneset, genesymbol
net = msigdb.rename(columns={"geneset": "source", "genesymbol": "target"})

# choose background
n_bg = adata.n_vars  # e.g. 33540
top_k = int(np.ceil(0.05 * n_bg))   # top 5%
n_up = n_bg - top_k                 # rank threshold so selected ~= top_k

dc.mt.ora(
    data=adata,
    net=net,
    tmin=5,
    raw=False,
    n_bg=n_bg,
    n_up=n_up,
    n_bm=0,
    verbose=True,
)

In [ ]:
acts = dc.get_acts(adata, obsm_key='ora_estimate')

# We need to remove inf and set them to the maximum value observed
acts_v = acts.X.ravel()
max_e = np.nanmax(acts_v[np.isfinite(acts_v)])
acts.X[~np.isfinite(acts.X)] = max_e

acts

In [ ]:
df = dc.rank_sources_groups(acts,groupby='celltype_coarse') 
n_markers = 4
source_markers =df.groupby('group').head(n_markers).groupby('group')['names'].apply(lambda x: list(x)).to_dict()

In [ ]:
sc.pl.matrixplot(acts, source_markers, 'celltype_coarse',
                 standard_scale='var',
                 colorbar_title='Z-scaled scores', cmap='RdBu_r', dendrogram=True,  save = "_hallmark_ploidy.pdf")

In [ ]:
df = dc.rank_sources_groups(acts, reference = "euploid", groupby='ploidy_xclone') 
n_markers = 4
source_markers =df.groupby('group').head(n_markers).groupby('group')['names'].apply(lambda x: list(x)).to_dict()


sc.pl.matrixplot(acts, source_markers, 'ploidy_xclone',
                 standard_scale='var',
                 colorbar_title='Z-scaled scores', cmap='RdBu_r', save = "_hallmark_ploidy.pdf")

In [ ]:
for cell_type in adata.obs['celltype_coarse'].unique() :
    
    adata_sub = adata[adata.obs['celltype_coarse'] == cell_type]
    
    dc.run_ora(
    mat=adata_sub,
    net=msigdb,
    source='geneset',
    target='genesymbol',
    verbose=True,
    use_raw=False
)
    acts = dc.get_acts(adata_sub, obsm_key='ora_estimate')

    # We need to remove inf and set them to the maximum value observed
    acts_v = acts.X.ravel()
    max_e = np.nanmax(acts_v[np.isfinite(acts_v)])
    
    acts.X[~np.isfinite(acts.X)] = max_e
    df = dc.rank_sources_groups(acts, reference = "euploid", groupby='ploidy_xclone') 
    n_markers = 5
    source_markers =df.groupby('group').head(n_markers).groupby('group')['names'].apply(lambda x: list(x)).to_dict()

    sc.pl.matrixplot(acts, source_markers, 'ploidy_xclone',
     standard_scale='var',
     title = cell_type, 
     show=False,
     colorbar_title='Z-scaled scores', cmap='RdBu_r', save = cell_type + "_hallmark_ploidy.pdf")

#### Further group based on ploidy percent

In [ ]:
adata.obs.tail()

In [ ]:
 plt.hist(adata.obs['per_complex'], bins=10, edgecolor='black')

In [ ]:
adata.obs['complexity'] = np.nan  # start with all NaNs
adata.obs.loc[adata.obs['per_complex'] < 50, 'complexity'] = 'low'
adata.obs.loc[adata.obs['per_complex'] >= 50, 'complexity'] = 'high'

In [ ]:
adata.obs['complexity'] = np.nan  # start with all NaNs
adata.obs.loc[adata.obs['per_complex'] <= 20, 'complexity'] = 'low'
adata.obs.loc[(adata.obs['per_complex'] > 20) & (adata.obs['per_complex'] < 60), 'complexity'] = 'medium'
adata.obs.loc[adata.obs['per_complex'] >= 60, 'complexity'] = 'high'

In [ ]:
adata_sub = adata[adata.obs['celltype_coarse'] == "unknown"]
adata_sub = adata_sub[adata_sub.obs['ploidy_xclone'] == "complex"]
adata_sub = adata_sub[~adata_sub.obs['complexity'].isna()].copy()
    
dc.run_ora(
mat=adata_sub,
net=msigdb,
source='geneset',
target='genesymbol',
verbose=True,
use_raw=False
)
acts = dc.get_acts(adata_sub, obsm_key='ora_estimate')

# We need to remove inf and set them to the maximum value observed
acts_v = acts.X.ravel()
max_e = np.nanmax(acts_v[np.isfinite(acts_v)])

acts.X[~np.isfinite(acts.X)] = max_e

In [ ]:
df = dc.rank_sources_groups(acts, groupby='complexity') 
n_markers = 5
source_markers =df.groupby('group').head(n_markers).groupby('group')['names'].apply(lambda x: list(x)).to_dict()

In [ ]:
sc.pl.matrixplot(acts, source_markers, 'complexity',
 standard_scale='var',
 show=True,
 colorbar_title='Z-scaled scores', cmap='RdBu_r', save =  "_hallmark_ploidy.pdf")

### GSEA

GSEA aggregates the per gene statistics across genes within a gene set, therefore making it possible to detect situations where all genes in a predefined set change in a small but coordinated way.  GSEA looks at the overall distribution of pathway genes within the ranked list to assess whether those pathways are more active or suppressed in your data. Are entire biological pathways are activated or repressed?